# 03 · Inference

Load the packaged pipeline (`models/model.joblib`) and score patients. The same
`src.predict.predict` function powers the API, so results here match the service exactly.

In [1]:
import sys, warnings
from pathlib import Path

here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "src" / "config.py").exists()), here)
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import json
import pandas as pd
from IPython.display import Image, display

from src import config
pd.set_option("display.max_columns", 30)

In [2]:
from src.predict import load_model, predict

model = load_model()
print(type(model).__name__, "->", [name for name, _ in model.steps])

Pipeline -> ['preprocess', 'model']


## Single patient (the sample from the API docs)

In [3]:
predict(model, config.SAMPLE_PATIENT)

[{'prediction': 0,
  'label': 'no heart disease',
  'confidence': 0.5402,
  'probability_disease': 0.4598}]

## Several patients from the cleaned dataset
*(Illustration only: the model has seen most of these rows during training.)*

In [4]:
sample = pd.read_csv(config.CLEAN_FILE).sample(8, random_state=0)
preds = pd.DataFrame(predict(model, sample[config.FEATURES].to_dict("records")), index=sample.index)
pd.concat([sample[["age", "sex", "cp", "thalach", "target"]], preds], axis=1).rename(columns={"target": "actual"})

,age,sex,cp,thalach,actual,prediction,label,confidence,probability_disease
225,34.0,0,2,192.0,0,0,no heart disease,0.9566,0.0434
152,67.0,0,3,160.0,0,0,no heart disease,0.5106,0.4894
228,54.0,1,4,108.0,1,1,heart disease,0.8369,0.8369
201,64.0,0,4,154.0,0,0,no heart disease,0.6449,0.3551
52,44.0,1,4,153.0,1,1,heart disease,0.5337,0.5337
245,67.0,1,4,71.0,1,1,heart disease,0.7101,0.7101
175,57.0,1,4,88.0,1,1,heart disease,0.9699,0.9699
168,35.0,1,4,156.0,1,1,heart disease,0.6570,0.6570


## Calling the running API
Start it with `uvicorn api.main:app` or Docker, then run the cell.

In [5]:
import json, urllib.request, urllib.error

def call_api(payload, url="http://localhost:8000/predict"):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return json.loads(r.read())
    except (urllib.error.URLError, OSError) as exc:
        return f"API not reachable ({exc}). Start it with: uvicorn api.main:app"

call_api(config.SAMPLE_PATIENT)

'API not reachable (<urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>). Start it with: uvicorn api.main:app'

## Input contract

The API rejects invalid input with HTTP 422 (see `PatientFeatures` in `api/main.py`).

In [6]:
pd.Series(config.FEATURE_DESCRIPTIONS, name='description').to_frame()

,description
age,Age in years
sex,"Sex (1 = male, 0 = female)"
cp,"Chest pain type (1 typical angina, 2 atypical ..."
trestbps,Resting blood pressure in mm Hg on admission
chol,Serum cholesterol in mg/dl
fbs,"Fasting blood sugar > 120 mg/dl (1 = true, 0 =..."
restecg,"Resting ECG result (0 normal, 1 ST-T abnormali..."
thalach,Maximum heart rate achieved
exang,"Exercise-induced angina (1 = yes, 0 = no)"
oldpeak,ST depression induced by exercise relative to ...
